# 05 Gas diffusion layer (IA-GDL-001)

**Requirement:** `IA-GDL-001`

Patents: US12308414B2, EP4602674A1. Equations are paraphrased; see claims/paragraphs cited below.

## Governing equations


Method-of-lines diffusion \(\partial_t c = \nabla\cdot(D_\mathrm{eff}\nabla c) - s_\mathrm{ORR}\).
\(D_\mathrm{eff} = D_\mathrm{gas}\,\varepsilon^{1.5}(1-s)^3\). Flooding (\(s\to 1\)) kills gas
diffusivity. Hydraulic head on vertical ORR drives liquid; OER bubbles on a horizontal
electrode can dry the TPB (US FIGS. 8-9B). pO2 0.01-100 atm discharge, 0.001-100 atm charge.


In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

here = Path.cwd()
for cand in (here, here.parent):
    if (cand / "src" / "ironair").is_dir() and (cand / "plant_sim").is_dir():
        sys.path.insert(0, str(cand))
        sys.path.insert(0, str(cand / "src"))
        break

from plant_sim.bootstrap import setup_path
setup_path()
from plant_sim.params import default_params, CAPACITY_FE_OH2_MAH_G, CAPACITY_MAGNETITE_MAH_G
from plant_sim.plotting import apply_style, timeseries, xy_plot, FIGURES
apply_style()
P = default_params()
print("6 M KOH default", P.c_KOH_mol_m3/1000, "M; T_ep", P.T_ep_sim_K, "K")
print("Faraday 960/320 checks", round(CAPACITY_FE_OH2_MAH_G,2), round(CAPACITY_MAGNETITE_MAH_G,2))
print("figures ->", FIGURES)


## Isolated component simulation

In [ ]:
from plant_sim.components.gdl import GasDiffusionLayer
from plant_sim.components.base import integrate_component
g = GasDiffusionLayer(P)
def u(t):
    return {"T_K": P.T_ep_sim_K, "P_Pa": P.P_atm_Pa, "x_O2": 0.21 if t<80 else 0.05, "r_orr_mol_s": 3e-7, "hydraulic_head_m": 0.05 if t<200 else 0.35, "oer_dryout": 0.0}
t, y = integrate_component(lambda t,y: g.rhs(t,y,u(t), {}), g.y0(), (0, 500), n_eval=160)
n = P.n_gdl_nodes
timeseries(t, {"air": y[0], "tpb": y[n-1], "s": y[n]}, xlabel="t (s)", ylabel="c_O2 / saturation", title="GDL O2 and flooding", callout="IA-GDL-001 delayed interior O2", stem="nb05_gdl")


## Verification against patents / SSS002

In [ ]:
print('Flooding reduces Deff; a boundary O2 step delays interior decline.')